## data preparation
### Erick Gordon

In [1]:
%pip install -U pandas openpyxl



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install xlrd


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install -U pandas


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import re
import unicodedata
import pandas as pd

def clean_names(cols):
    out = []
    for c in cols:
        c = str(c).strip().lower()
        # quitar acentos (á -> a, ñ -> n)
        c = unicodedata.normalize("NFKD", c).encode("ascii", "ignore").decode("ascii")
        # cualquier cosa que no sea [a-z0-9] -> _
        c = re.sub(r"[^a-z0-9]+", "_", c)
        c = re.sub(r"_+", "_", c).strip("_")
        out.append(c if c else "x")
    # asegurar unicidad si hay repetidos
    seen = {}
    uniq = []
    for c in out:
        if c not in seen:
            seen[c] = 0
            uniq.append(c)
        else:
            seen[c] += 1
            uniq.append(f"{c}_{seen[c]}")
    return uniq

In [5]:
import pandas as pd

ruta = "00_data/Tasas_Creditos.xlsx"

df = pd.read_excel(
    ruta,
    sheet_name="CONSUMO",
    engine="openpyxl"
)

# convertir la 1ra columna a fecha (sin asumir el nombre)
col_fecha = df.columns[0]
df[col_fecha] = pd.to_datetime(df[col_fecha], errors="coerce")

# 2) Última columna = numérica
col_num = df.columns[-1]

# Si viene como string con comas (ej. "1,234.56" o "1.234,56"), limpia lo más común:
s = df[col_num].astype(str).str.strip()

# caso típico: separador miles "," y decimal "." -> "1,234.56"
s1 = s.str.replace(",", "", regex=False)

# intenta convertir
df[col_num] = pd.to_numeric(s1, errors="coerce")
df.columns = clean_names(df.columns)




In [6]:
df.head()

,fecha,tarjeta_de_credito,prestamo_personal,auto,vivienda_no_preferencial
0,2001-12-01,19.335481,11.642732,11.084210,NaN
1,2002-01-01,19.281838,11.260737,11.461143,NaN
2,2002-02-01,19.285959,11.659333,11.481690,NaN
3,2002-03-01,19.365961,11.168228,11.458394,NaN
4,2002-04-01,19.34,11.238345,11.438697,NaN


In [7]:
# df.to_csv("00_data/datos_limpios.csv", index=True, encoding="utf-8", sep=",")

In [8]:
#%pip install -U pyarrow
#df.to_parquet("00_data/datos_limpios.parquet", index=True, engine="pyarrow")

In [9]:
%pip install -U plotly


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
%pip install -U nbformat ipython


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
import pandas as pd
import plotly.graph_objects as go

df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
df = df.sort_values("fecha")

fig = go.Figure()

# Eje izquierdo (y)
fig.add_trace(go.Scatter(
    x=df["fecha"], y=df["prestamo_personal"],
    mode="lines", name="prestamo_personal",
    line=dict(color="#b2182b"),
    yaxis="y"
))

fig.add_trace(go.Scatter(
    x=df["fecha"], y=df["auto"],
    mode="lines", name="auto",
    line=dict(color="#4d4d4d"),
    yaxis="y"
))

fig.add_trace(go.Scatter(
    x=df["fecha"], y=df["vivienda_no_preferencial"],
    mode="lines", name="vivienda_no_preferencial",
    line=dict(color="#1a9850"),
    yaxis="y"
))

# Eje derecho (y2): tarjeta_de_credito
fig.add_trace(go.Scatter(
    x=df["fecha"], y=df["tarjeta_de_credito"],
    mode="lines", name="tarjeta_de_credito",
    line=dict(color="#2166ac"),
    yaxis="y2"
))

fig.update_layout(
    title="Series por fecha (tarjeta_de_credito en eje derecho)",
    xaxis=dict(title="Fecha"),
    yaxis=dict(title="Valor (izquierdo)"),
    yaxis2=dict(
        title="tarjeta_de_credito (derecho)",
        overlaying="y",
        side="right"
    ),
    hovermode="x unified",
    template="plotly_white"
)

fig.show()


## datos de la FED

In [15]:
%pip install python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [12]:
# Instala dependencias (si hace falta)
%pip install -U pandas pandas_datareader


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Si te faltan dependencias:
# %pip install -U pandas requests

import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
# 
FRED_API_KEY = os.getenv("FRED_API_KEY")  # exporta tu key en la variable de entorno
if not FRED_API_KEY:
    raise RuntimeError("Define tu API key en la variable de entorno FRED_API_KEY")


SERIES = {
    "fed_funds_eff": "DFF", # Federal Funds Effective Rate. Tasa efectiva promedio ponderada de las operaciones overnight en el mercado interbancario de reservas.
    "ust_3m": "DGS3MO",     # 3-Month Treasury Constant Maturity Rate. Rendimiento del bono del Tesoro estadounidense a 3 meses (constant maturity).
    "ust_1y": "DGS1",       # 1-Year Treasury Constant Maturity Rate. Rendimiento del Treasury a 1 año
    "ust_3y": "DGS3",       # 3-Year Treasury Constant Maturity Rate. Rendimiento del Treasury a 3 años
    "ust_5y": "DGS5",       # Idem
    "ust_10y": "DGS10",     # Idem
    "ust_30y": "DGS30",     # Idem
}


def fred_observations(series_id: str, api_key: str, start: str | None = None, end: str | None = None) -> pd.Series:
    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
        "series_id": series_id,
        "api_key": api_key,
        "file_type": "json",
    }
    if start:
        params["observation_start"] = start
    if end:
        params["observation_end"] = end

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    js = r.json()

    obs = pd.DataFrame(js["observations"])
    obs["date"] = pd.to_datetime(obs["date"], errors="coerce")

    # FRED usa "." para faltantes
    obs["value"] = pd.to_numeric(obs["value"].replace(".", pd.NA), errors="coerce")

    s = obs.set_index("date")["value"].sort_index()
    s.name = series_id
    return s

import requests

def fred_series_meta(series_id: str, api_key: str) -> dict:
    url = "https://api.stlouisfed.org/fred/series"
    params = {"series_id": series_id, "api_key": api_key, "file_type": "json"}

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    js = r.json()

    # FRED típicamente: {"seriess":[{...}]}
    if isinstance(js.get("seriess"), list) and len(js["seriess"]) > 0:
        ser = js["seriess"][0]
    # fallback por si viniera con otra forma
    elif isinstance(js.get("seriess"), dict) and "series" in js["seriess"] and len(js["seriess"]["series"]) > 0:
        ser = js["seriess"]["series"][0]
    else:
        raise ValueError(f"No se encontró metadata para series_id={series_id}. Respuesta: {js}")

    return {
        "series_id": ser.get("id"),
        "title": ser.get("title"),
        "frequency": ser.get("frequency"),
        "frequency_short": ser.get("frequency_short"),
        "units": ser.get("units"),
        "last_updated": ser.get("last_updated"),
        "observation_start": ser.get("observation_start"),
        "observation_end": ser.get("observation_end"),
    }

# --- Descarga de datos (observaciones) ---
start = "2000-01-01"   # ajusta a gusto
end = None             # o "2026-01-01"

df_rates = pd.concat(
    [fred_observations(code, FRED_API_KEY, start=start, end=end).rename(name)
     for name, code in SERIES.items()],
    axis=1
)

# (opcional) ordenar y dejar índice diario
df_rates = df_rates.sort_index()

df_rates.head()





,fed_funds_eff,ust_3m,ust_1y,ust_3y,ust_5y,ust_10y,ust_30y
date,,,,,,,
2000-01-01,3.99,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-02,3.99,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-03,5.43,5.48,6.09,6.42,6.50,6.58,6.61
2000-01-04,5.38,5.43,6.00,6.34,6.40,6.49,6.53
2000-01-05,5.41,5.44,6.05,6.43,6.51,6.62,6.64


In [20]:
df_rates.head()

,fed_funds_eff,ust_3m,ust_1y,ust_3y,ust_5y,ust_10y,ust_30y
date,,,,,,,
2000-01-01,3.99,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-02,3.99,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-03,5.43,5.48,6.09,6.42,6.50,6.58,6.61
2000-01-04,5.38,5.43,6.00,6.34,6.40,6.49,6.53
2000-01-05,5.41,5.44,6.05,6.43,6.51,6.62,6.64


In [21]:
# --- (Opcional) Metadatos para ver frecuencia/last_updated ---
df_meta = pd.DataFrame([fred_series_meta(code, FRED_API_KEY) for code in SERIES.values()])
df_meta

,series_id,title,frequency,frequency_short,units,last_updated,observation_start,observation_end
0,DFF,Federal Funds Effective Rate,"Daily, 7-Day",D,Percent,2026-02-13 15:16:43-06,1954-07-01,2026-02-12
1,DGS3MO,Market Yield on U.S. Treasury Securities at 3-...,Daily,D,Percent,2026-02-13 15:17:00-06,1981-09-01,2026-02-12
2,DGS1,Market Yield on U.S. Treasury Securities at 1-...,Daily,D,Percent,2026-02-13 15:17:03-06,1962-01-02,2026-02-12
3,DGS3,Market Yield on U.S. Treasury Securities at 3-...,Daily,D,Percent,2026-02-13 15:16:58-06,1962-01-02,2026-02-12
4,DGS5,Market Yield on U.S. Treasury Securities at 5-...,Daily,D,Percent,2026-02-13 15:17:02-06,1962-01-02,2026-02-12
5,DGS10,Market Yield on U.S. Treasury Securities at 10...,Daily,D,Percent,2026-02-13 15:16:50-06,1962-01-02,2026-02-12
6,DGS30,Market Yield on U.S. Treasury Securities at 30...,Daily,D,Percent,2026-02-13 15:17:05-06,1977-02-15,2026-02-12


## une ambos datos

In [23]:
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce").dt.normalize()

df_rates2 = df_rates.copy()
df_rates2.index = pd.to_datetime(df_rates2.index).normalize()

# (opcional) ffill de tasas
df_rates2 = df_rates2.sort_index().ffill().reset_index().rename(columns={"date": "fecha"})
df_rates2


,fecha,fed_funds_eff,ust_3m,ust_1y,ust_3y,ust_5y,ust_10y,ust_30y
0,2000-01-01,3.99,NaN,NaN,NaN,NaN,NaN,NaN
1,2000-01-02,3.99,NaN,NaN,NaN,NaN,NaN,NaN
2,2000-01-03,5.43,5.48,6.09,6.42,6.50,6.58,6.61
3,2000-01-04,5.38,5.43,6.00,6.34,6.40,6.49,6.53
4,2000-01-05,5.41,5.44,6.05,6.43,6.51,6.62,6.64
...,...,...,...,...,...,...,...,...
9535,2026-02-08,3.64,3.68,3.45,3.57,3.76,4.22,4.85
9536,2026-02-09,3.64,3.69,3.43,3.56,3.75,4.22,4.85
9537,2026-02-10,3.64,3.69,3.40,3.50,3.70,4.16,4.78
9538,2026-02-11,3.64,3.70,3.47,3.55,3.75,4.18,4.82


In [24]:

df_join = df.merge(df_rates2, on="fecha", how="left").sort_values("fecha")
df_join = df_join.dropna(subset=["fecha"])

df_join.sort_index(ascending=False).head()

,fecha,tarjeta_de_credito,prestamo_personal,auto,vivienda_no_preferencial,fed_funds_eff,ust_3m,ust_1y,ust_3y,ust_5y,ust_10y,ust_30y
287,2025-11-01,22.05,8.92,7.93,6.25,3.86,3.89,3.70,3.60,3.71,4.11,4.67
286,2025-10-01,22.02,8.92,7.94,6.24,4.09,4.01,3.62,3.56,3.68,4.12,4.72
285,2025-09-01,21.99,8.91,7.96,6.24,4.33,4.23,3.83,3.58,3.68,4.23,4.92
284,2025-08-01,21.94,8.90,7.95,6.24,4.33,4.35,3.87,3.67,3.77,4.23,4.81
283,2025-07-01,21.94,8.91,7.95,6.25,4.33,4.40,3.98,3.75,3.84,4.26,4.78


## correlacion

In [25]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

import plotly.io as pio
pio.renderers.default = "browser"

num = df_join.select_dtypes(include=[np.number]).copy()
num = num.dropna(axis=1, how="all")
num = num.loc[:, num.nunique(dropna=True) > 1]

corr = num.corr(method="pearson")

fig = go.Figure(
    data=go.Heatmap(
        z=corr.values,
        x=corr.columns.tolist(),
        y=corr.index.tolist(),
        zmin=-1, zmax=1,
        colorscale="RdBu",
        reversescale=True,
        text=np.round(corr.values, 2),
        texttemplate="%{text}",
        hovertemplate="X=%{x}<br>Y=%{y}<br>corr=%{z:.3f}<extra></extra>",
    )
)

fig.update_layout(
    title="Matriz de correlación (Pearson)",
    xaxis_title="",
    yaxis_title="",
    template="plotly_white"
)

fig.show()


In [ ]:
# df_join.to_csv("00_data/rates_unified.csv", index=True, encoding="utf-8", sep=",")